In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2
%matplotlib inline


The lab_black extension is already loaded. To reload it, use:
  %reload_ext lab_black
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import os
import pickle
import numpy as np
from mne_connectivity.viz import plot_connectivity_circle
from mne.viz import circular_layout
import matplotlib.pyplot as plt
import numpy as np
import pickle


def create_hierarchy():
    hierarchy = {}
    fmin_fmax = [(1, 4), (4, 8), (8, 12), (13, 30), (30, 50), (1, 120)]
    conditions = range(1, 7)
    patients = range(2, 22)
    # states = ["pre", "during", "post"]
    states = ["pre"]

    for fmin, fmax in fmin_fmax:
        filter_key = f"{fmin}_{fmax}"
        hierarchy[filter_key] = {}
        for condition in conditions:
            hierarchy[filter_key][condition] = {}
            for patient in patients:
                hierarchy[filter_key][condition][patient] = {}
                folder = f"D:\\neuro-closeloop-project\\vielight_close_loop\\clean_notebooks\\imcoh\\csd\\filters\\{filter_key}\\conditions\\{condition}\\results\\{patient}"
                if os.path.exists(folder) and os.listdir(folder):
                    for state in states:
                        pickle_file = os.path.join(
                            folder, f"{patient}_{state}_connectivity.pkl"
                        )
                        if os.path.exists(pickle_file):
                            with open(pickle_file, "rb") as f:
                                hierarchy[filter_key][condition][patient][state] = (
                                    pickle.load(f)
                                )
                        else:
                            print(f"Missing pickle file: {pickle_file}")
                            hierarchy[filter_key][condition][patient][state] = None
                else:
                    print(f"Empty or non-existent folder: {folder}")
                    for state in states:
                        hierarchy[filter_key][condition][patient][state] = None

    return hierarchy


In [6]:
# Usage
hierarchy = create_hierarchy()
print(hierarchy.keys())
print(hierarchy["1_120"].keys())
print(hierarchy["1_120"][1].keys())
print(hierarchy["1_120"][1][2].keys())
print(hierarchy["1_120"][1][2]["pre"])


Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\1\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\2\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\3\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\4\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\5\results\13
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\6\results\2
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\6\results\6
Empty or non-existent folder:

In [8]:
print(hierarchy["1_120"][1][2]["pre"].get_data("dense").shape)


(60, 60, 1)


# Visualization

In [ ]:
def get_channel_5_colors():
    return [
        "AF3",
        "F1",
        "F3",
        "F5",
        "F7",
        "FC1",
        "FC3",
        "FC5",
        "FCZ",
        "FP1",
        "FPZ",
        "FT7",
        "FZ",
        "C1",
        "C3",
        "C5",
        "CZ",
        "T7",
        "TP7",
        "CP1",
        "CP3",
        "CP5",
        "P1",
        "P3",
        "P5",
        "P7",
        "PO3",
        "PO5",
        "PO7",
        "O1",
        "POZ",
        "OZ",
        "PZ",
        "CPZ",
        "TP8",
        "T8",
        "PO8",
        "PO6",
        "PO4",
        "O2",
        "P8",
        "P6",
        "P4",
        "P2",
        "CP6",
        "CP4",
        "CP2",
        "C6",
        "C4",
        "C2",
        "FT8",
        "FP2",
        "FC6",
        "FC4",
        "FC2",
        "F8",
        "F6",
        "F4",
        "F2",
        "AF4",
    ]


def assign_channel_5_colors():
    red = (0.458, 0.0, 0.0, 1.0)
    green = (0.0, 1.0, 0.0, 1.0)
    blue = (0.0, 0.0, 0.999, 1.0)
    purple = (0.5, 0.0, 0.5, 1.0)
    orange = (1.0, 0.647, 0.0, 1.0)
    black = (0.0, 0.0, 0.0, 1.0)

    # New color for separators
    channel_color = {
        # Frontal (Red)
        "AF3": red,
        "AF4": red,
        "F1": red,
        "F2": red,
        "F3": red,
        "F4": red,
        "F5": red,
        "F6": red,
        "F7": red,
        "F8": red,
        "FC1": red,
        "FC2": red,
        "FC3": red,
        "FC4": red,
        "FC5": red,
        "FC6": red,
        "FCZ": red,
        "FP1": red,
        "FP2": red,
        "FPZ": red,
        "FT7": red,
        "FT8": red,
        "FZ": red,
        # Central (Green)
        "C1": green,
        "C2": green,
        "C3": green,
        "C4": green,
        "C5": green,
        "C6": green,
        "CZ": green,
        # Parietal (Blue)
        "CP1": blue,
        "CP2": blue,
        "CP3": blue,
        "CP4": blue,
        "CP5": blue,
        "CP6": blue,
        "CPZ": blue,
        "P1": blue,
        "P2": blue,
        "P3": blue,
        "P4": blue,
        "P5": blue,
        "P6": blue,
        "P7": blue,
        "P8": blue,
        "PZ": blue,
        # Occipital (Purple)
        "O1": purple,
        "O2": purple,
        "OZ": purple,
        "PO3": purple,
        "PO4": purple,
        "PO5": purple,
        "PO6": purple,
        "PO7": purple,
        "PO8": purple,
        "POZ": purple,
        # Temporal (Orange)
        "T7": orange,
        "T8": orange,
        "TP7": orange,
        "TP8": orange,
    }

    return channel_color


def prepare_circular_layout(label_names):
    """
    Prepare the circular layout for the connectivity plot.

    Args:
    label_names (list): List of channel names.

    Returns:
    tuple: Node angles and node order for the circular layout.
    """
    n_channels = len(label_names)
    node_angles = circular_layout(
        label_names,
        label_names,
        start_pos=90,
        group_boundaries=[0, len(label_names) // 2],
    )
    node_order = label_names
    return node_angles, node_order


def get_circular_plot_requirements(label_names):
    """
    Get the requirements for a circular connectivity plot with left channels first.

    Args:
    label_names (list): List of channel names.

    Returns:
    tuple: Node angles, node order, and node colors for the circular plot.
    """

    sorted_channels = get_channel_5_colors()

    # Assign colors to channels
    # channel_color = assign_channel_colors(channels_side, left_color, right_color)
    channel_color = assign_channel_5_colors()

    # Prepare circular layout with sorted channels
    node_angles, _ = prepare_circular_layout(list(channel_color.keys()))

    # Get node colors in the same order as sorted_channels
    node_colors = [channel_color[channel] for channel in sorted_channels]

    return node_angles, sorted_channels, node_colors


def plot_circular_connectivity(
    connectivity,
    stage,
    label_names,
    node_angles,
    node_colors,
    method="wpli",
    is_save=False,
    n_lines=20,
    figsize=(8, 8),
):
    """
    Plot circular connectivity diagram.

    Args:
    connectivity (mne.Connectivity): The connectivity object.
    stage (str): The stage of the experiment (e.g., 'pre', 'during', 'post').
    label_names (list): List of channel names.
    node_angles (dict): Dictionary of node angles for the circular plot.
    node_colors (list): List of colors for each node.
    method (str, optional): Connectivity method used. Defaults to "wpli".
    is_save (bool, optional): Whether to save the figure. Defaults to False.
    n_lines (int, optional): Number of connections to draw. Defaults to 20.
    figsize (tuple, optional): Figure size. Defaults to (8, 8).

    Returns:
    matplotlib.figure.Figure: The created figure.
    """
    # Create figure
    fig, ax = plt.subplots(
        figsize=figsize, facecolor="black", subplot_kw=dict(polar=True)
    )

    if not isinstance(connectivity, np.ndarray):
        connectivity = connectivity.get_data("dense")[:, :, 0]

    if connectivity.ndim == 3:
        connectivity = connectivity[:, :, 0]

    print(f"{len(node_angles) = }")
    print(f"{len(node_colors) = }")

    # Plot connectivity circle
    plot_connectivity_circle(
        con=connectivity,
        node_names=label_names,
        n_lines=n_lines,
        node_angles=node_angles,
        node_colors=node_colors,
        title=f"All-to-All Connectivity {stage} stimuli Condition ({method})",
        ax=ax,
    )

    # Adjust layout
    fig.tight_layout()

    # Save figure if requested
    if is_save:
        filename = f"{method}_{stage}.png"
        fig.savefig(filename, dpi=300)
        print(f"Figure saved as {filename}")

    return fig

def prepare_plot_data(hierarchy, freq_band, condition, patient, state):
    """
    Extract data from the hierarchy and prepare for plotting.

    Args:
    hierarchy (dict): The hierarchical data structure
    freq_band (str): Frequency band key (e.g., '1_4', '4_8', etc.)
    condition (int): Condition number (1-6)
    patient (int): Patient number (2-21)
    state (str): State ('pre', 'during', 'post')

    Returns:
    tuple: connectivity_data, node_angles, node_order, node_colors
    """
    # Extract SpectralConnectivity object
    spectral_conn = hierarchy[freq_band][condition][patient][state]

    if spectral_conn is None:
        raise ValueError(
            f"No data for {freq_band}, condition {condition}, patient {patient}, state {state}"
        )

    # Extract connectivity data
    connectivity_data = spectral_conn.get_data(output="dense")[
        :, :, 0
    ]  # Assuming first frequency

    # Get channel names
    channel_names = get_channel_5_colors()

    # Prepare circular layout
    node_angles, node_order, node_colors = get_circular_plot_requirements(channel_names)

    return connectivity_data, node_angles, node_order, node_colors


In [ ]:
# 6 conditions vs 5 filters


def plot_comprehensive_connectivity(hierarchy, patient=2, state="pre"):
    """
    Create a comprehensive plot of connectivity for all conditions and frequency bands.

    Args:
    hierarchy (dict): The hierarchical data structure
    patient (int): Patient number (default: 2)
    state (str): State to plot (default: 'pre')

    Returns:
    matplotlib.figure.Figure: The created figure
    """
    freq_bands = ["1_4", "4_8", "8_12", "13_30", "30_50", "1_120"]
    conditions = range(1, 7)

    fig, axes = plt.subplots(6, 6, figsize=(25, 30), subplot_kw=dict(polar=True))
    fig.suptitle(
        f"Connectivity for Patient {patient} - {state.capitalize()} State", fontsize=16
    )

    for i, condition in enumerate(conditions):
        for j, freq_band in enumerate(freq_bands):
            ax = axes[i, j]

            try:
                connectivity_data, node_angles, node_order, node_colors = (
                    prepare_plot_data(hierarchy, freq_band, condition, patient, state)
                )

                plot_connectivity_circle(
                    con=connectivity_data,
                    node_names=node_order,
                    n_lines=20,  # Adjust this value as needed
                    node_angles=node_angles,
                    node_colors=node_colors,
                    title=f"Condition {condition}, {freq_band} Hz",
                    ax=ax,
                    show=False,
                )

                # Remove the color bar for each subplot to save space
                ax.collections[-1].colorbar.remove()

            except Exception as e:
                ax.text(0.5, 0.5, f"No data\n{str(e)}", ha="center", va="center")

            # Set the title for each subplot
            ax.set_title(f"Condition {condition}, {freq_band} Hz", fontsize=10)

    # Add row and column labels
    for i, condition in enumerate(conditions):
        fig.text(
            0.08, 0.85 - i * 0.14, f"Condition {condition}", rotation=90, va="center"
        )

    for j, freq_band in enumerate(freq_bands):
        fig.text(0.15 + j * 0.17, 0.08, f"{freq_band} Hz", ha="center")

    plt.tight_layout()
    plt.subplots_adjust(top=0.95, bottom=0.1, left=0.1, right=0.95)

    return fig


def save_comprehensive_connectivity_plots(hierarchy, base_dir="connectivity_plots"):
    """
    Create and save comprehensive connectivity plots for all patients and states.

    Args:
    hierarchy (dict): The hierarchical data structure
    base_dir (str): Base directory to save the plots (default: 'connectivity_plots')
    """
    freq_bands = ["1_4", "4_8", "8_12", "13_30", "30_50", "1_120"]
    conditions = range(1, 7)
    states = ["pre", "during", "post"]
    patients = range(2, 22)  # Assuming patients are numbered from 2 to 21

    for patient in patients:
        for state in states:
            print(f"Processing Patient {patient}, State: {state}")

            fig, axes = plt.subplots(
                6, 6, figsize=(25, 30), subplot_kw=dict(polar=True)
            )
            fig.suptitle(
                f"Connectivity for Patient {patient} - {state.capitalize()} State",
                fontsize=16,
            )

            for i, condition in enumerate(conditions):
                for j, freq_band in enumerate(freq_bands):
                    ax = axes[i, j]

                    try:
                        connectivity_data, node_angles, node_order, node_colors = (
                            prepare_plot_data(
                                hierarchy, freq_band, condition, patient, state
                            )
                        )

                        plot_connectivity_circle(
                            con=connectivity_data,
                            node_names=node_order,
                            n_lines=20,  # Adjust this value as needed
                            node_angles=node_angles,
                            node_colors=node_colors,
                            title=f"Condition {condition}, {freq_band} Hz",
                            ax=ax,
                            show=False,
                        )

                        # Remove the color bar for each subplot to save space
                        ax.collections[-1].colorbar.remove()

                    except Exception as e:
                        ax.text(
                            0.5, 0.5, f"No data\n{str(e)}", ha="center", va="center"
                        )

                    # Set the title for each subplot
                    ax.set_title(f"Condition {condition}, {freq_band} Hz", fontsize=10)

            # Add row and column labels
            for i, condition in enumerate(conditions):
                fig.text(
                    0.08,
                    0.85 - i * 0.14,
                    f"Condition {condition}",
                    rotation=90,
                    va="center",
                )

            for j, freq_band in enumerate(freq_bands):
                fig.text(0.15 + j * 0.17, 0.08, f"{freq_band} Hz", ha="center")

            plt.tight_layout()
            plt.subplots_adjust(top=0.95, bottom=0.1, left=0.1, right=0.95)

            # Create directory if it doesn't exist
            save_dir = os.path.join(base_dir, f"patient_{patient}")
            os.makedirs(save_dir, exist_ok=True)

            # Save the figure
            filename = os.path.join(
                save_dir, f"connectivity_plot_patient_{patient}_{state}.png"
            )
            fig.savefig(filename, dpi=300, bbox_inches="tight")
            plt.close(fig)  # Close the figure to free up memory

            print(f"Saved plot to {filename}")


# Usage
# fig = plot_comprehensive_connectivity(hierarchy, patient=2, state="pre")
# plt.show()

# save_comprehensive_connectivity_plots(
#     hierarchy, base_dir="csd_results/filter_banks_reuslts/conditions_vs_filters"
# )


In [ ]:
def average_connectivity_plot(hierarchy, base_dir="average_connectivity_plots"):
    """
    Create and save comprehensive connectivity plots averaged across all patients.

    Args:
    hierarchy (dict): The hierarchical data structure
    base_dir (str): Base directory to save the plots (default: 'average_connectivity_plots')
    """
    freq_bands = ["1_4", "4_8", "8_12", "13_30", "30_50", "1_120"]
    conditions = range(1, 7)
    states = ["pre", "during", "post"]
    patients = range(2, 22)  # Assuming patients are numbered from 2 to 21

    os.makedirs(base_dir, exist_ok=True)

    for state in states:
        print(f"Processing State: {state}")

        fig, axes = plt.subplots(6, 6, figsize=(25, 30), subplot_kw=dict(polar=True))
        fig.suptitle(
            f"Average Connectivity Across Patients - {state.capitalize()} State",
            fontsize=16,
        )

        for i, condition in enumerate(conditions):
            for j, freq_band in enumerate(freq_bands):
                ax = axes[i, j]

                try:
                    # Initialize an array to store connectivity data for all patients
                    all_patient_data = []

                    for patient in patients:
                        try:
                            connectivity_data, node_angles, node_order, node_colors = (
                                prepare_plot_data(
                                    hierarchy, freq_band, condition, patient, state
                                )
                            )
                            all_patient_data.append(connectivity_data)
                        except Exception as e:
                            print(f"Error processing patient {patient}: {str(e)}")

                    # Calculate the average connectivity across patients
                    if all_patient_data:
                        avg_connectivity = np.mean(all_patient_data, axis=0)

                        plot_connectivity_circle(
                            con=avg_connectivity,
                            node_names=node_order,
                            n_lines=20,  # Adjust this value as needed
                            node_angles=node_angles,
                            node_colors=node_colors,
                            title=f"Condition {condition}, {freq_band} Hz",
                            ax=ax,
                            show=False,
                        )

                        # Remove the color bar for each subplot to save space
                        ax.collections[-1].colorbar.remove()
                    else:
                        ax.text(0.5, 0.5, "No data available", ha="center", va="center")

                except Exception as e:
                    ax.text(0.5, 0.5, f"Error: {str(e)}", ha="center", va="center")

                # Set the title for each subplot
                ax.set_title(f"Condition {condition}, {freq_band} Hz", fontsize=10)

        # Add row and column labels
        for i, condition in enumerate(conditions):
            fig.text(
                0.08,
                0.85 - i * 0.14,
                f"Condition {condition}",
                rotation=90,
                va="center",
            )

        for j, freq_band in enumerate(freq_bands):
            fig.text(0.15 + j * 0.17, 0.08, f"{freq_band} Hz", ha="center")

        plt.tight_layout()
        plt.subplots_adjust(top=0.95, bottom=0.1, left=0.1, right=0.95)

        # Save the figure
        filename = os.path.join(base_dir, f"average_connectivity_plot_{state}.png")
        fig.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)  # Close the figure to free up memory

        print(f"Saved plot to {filename}")


# Usage
average_connectivity_plot(
    hierarchy,
    base_dir="csd_results/filter_banks_reuslts/conditions_vs_filters_average_across_patients",
)


In [ ]:
def condition_based_average_connectivity_plot(
    hierarchy, base_dir="condition_based_connectivity_plots"
):
    """
    Create and save connectivity plots for each condition, showing all states and frequency bands,
    averaged across all patients.

    Args:
    hierarchy (dict): The hierarchical data structure
    base_dir (str): Base directory to save the plots (default: 'condition_based_connectivity_plots')
    """
    freq_bands = ["1_4", "4_8", "8_12", "13_30", "30_50", "1_120"]
    conditions = range(1, 7)
    states = ["pre", "during", "post"]
    patients = range(2, 22)  # Assuming patients are numbered from 2 to 21

    os.makedirs(base_dir, exist_ok=True)

    for condition in conditions:
        print(f"Processing Condition: {condition}")

        fig, axes = plt.subplots(3, 6, figsize=(25, 15), subplot_kw=dict(polar=True))
        fig.suptitle(
            f"Average Connectivity Across Patients - Condition {condition}", fontsize=16
        )

        for i, state in enumerate(states):
            for j, freq_band in enumerate(freq_bands):
                ax = axes[i, j]

                try:
                    # Initialize an array to store connectivity data for all patients
                    all_patient_data = []

                    for patient in patients:
                        try:
                            connectivity_data, node_angles, node_order, node_colors = (
                                prepare_plot_data(
                                    hierarchy, freq_band, condition, patient, state
                                )
                            )
                            all_patient_data.append(connectivity_data)
                        except Exception as e:
                            print(f"Error processing patient {patient}: {str(e)}")

                    # Calculate the average connectivity across patients
                    if all_patient_data:
                        avg_connectivity = np.mean(all_patient_data, axis=0)

                        plot_connectivity_circle(
                            con=avg_connectivity,
                            node_names=node_order,
                            n_lines=20,  # Adjust this value as needed
                            node_angles=node_angles,
                            node_colors=node_colors,
                            title=f"{state.capitalize()}, {freq_band} Hz",
                            ax=ax,
                            show=False,
                        )

                        # Remove the color bar for each subplot to save space
                        ax.collections[-1].colorbar.remove()
                    else:
                        ax.text(0.5, 0.5, "No data available", ha="center", va="center")

                except Exception as e:
                    ax.text(0.5, 0.5, f"Error: {str(e)}", ha="center", va="center")

                # Set the title for each subplot
                ax.set_title(f"{state.capitalize()}, {freq_band} Hz", fontsize=10)

        # Add row and column labels
        for i, state in enumerate(states):
            fig.text(0.08, 0.75 - i * 0.3, state.capitalize(), rotation=90, va="center")

        for j, freq_band in enumerate(freq_bands):
            fig.text(0.15 + j * 0.17, 0.08, f"{freq_band} Hz", ha="center")

        plt.tight_layout()
        plt.subplots_adjust(top=0.92, bottom=0.1, left=0.1, right=0.95)

        # Save the figure
        filename = os.path.join(
            base_dir, f"average_connectivity_plot_condition_{condition}.png"
        )
        fig.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)  # Close the figure to free up memory

        print(f"Saved plot to {filename}")


# Usage
condition_based_average_connectivity_plot(
    hierarchy,
    base_dir="csd_results/filter_banks_reuslts/state_vs_filters_average_across_patients",
)


In [11]:
import os
import pickle
import numpy as np
from mne_connectivity.viz import plot_connectivity_circle
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg


def create_connectivity_plots(hierarchy, state: str = "pre",output_dir="imcoh_pre_consistency_csd"):
    """
    Creates connectivity plots for all patients, conditions, and frequency bands.
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Frequency bands
    freq_bands = [(1, 4), (4, 8), (8, 12), (13, 30), (30, 50), (1, 120)]

    # Get channel names and layout requirements
    channel_names = get_channel_5_colors()

    # Process each patient
    for patient in range(2, 22):
        # Create figure with subplots
        fig, axes = plt.subplots(6, 6, figsize=(30, 30))
        fig.suptitle(f"Patient {patient} Connectivity Analysis", fontsize=16)

        # Process each frequency band (columns)
        for freq_idx, (fmin, fmax) in enumerate(freq_bands):
            freq_key = f"{fmin}_{fmax}"

            # Add frequency band labels
            axes[0, freq_idx].set_title(f"{fmin}-{fmax} Hz")

            # Process each condition (rows)
            for condition in range(1, 7):
                try:
                    # Prepare plot data
                    connectivity_data, node_angles, node_order, node_colors = (
                        prepare_plot_data(
                            hierarchy, freq_key, condition, patient, state
                        )
                    )

                    # Create connectivity plot with explicit canvas
                    fig_conn = plt.figure(figsize=(8, 8), facecolor="black")
                    canvas = FigureCanvasAgg(fig_conn)
                    ax = fig_conn.add_subplot(111, polar=True)

                    # Plot connectivity circle
                    plot_connectivity_circle(
                        con=connectivity_data,
                        node_names=node_order,
                        node_angles=node_angles,
                        node_colors=node_colors,
                        title=f"Condition {condition}, {fmin}-{fmax} Hz",
                        ax=ax,
                        show=False,
                        n_lines=20,
                    )

                    # Draw the canvas before accessing the renderer
                    canvas.draw()

                    # Convert to image array
                    img_array = np.asarray(canvas.buffer_rgba())

                    # Display in the grid
                    axes[condition - 1, freq_idx].imshow(img_array)

                    # Clean up
                    plt.close(fig_conn)

                except Exception as e:
                    print(
                        f"Error processing patient {patient}, condition {condition}, "
                        f"frequency {freq_key}: {str(e)}"
                    )
                    axes[condition - 1, freq_idx].text(
                        0.5, 0.5, "Error", ha="center", va="center"
                    )

                # Remove axis ticks
                axes[condition - 1, freq_idx].set_xticks([])
                axes[condition - 1, freq_idx].set_yticks([])

        # Add row labels
        for condition in range(1, 7):
            axes[condition - 1, 0].set_ylabel(f"Condition {condition}")

        # Adjust layout and save
        plt.tight_layout()
        output_file = os.path.join(
            output_dir, f"patient_{patient}_connectivity_grid.png"
        )
        plt.savefig(output_file, dpi=300, bbox_inches="tight")
        plt.close("all")  # Close all figures to prevent memory issues

        print(f"Completed processing patient {patient}")


# Create the hierarchy
hierarchy = create_hierarchy()

# Set matplotlib backend to Agg
plt.switch_backend("Agg")

# Generate and save plots
create_connectivity_plots(hierarchy)


Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\1\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\2\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\3\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\4\results\16
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\5\results\13
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\6\results\2
Empty or non-existent folder: D:\neuro-closeloop-project\vielight_close_loop\clean_notebooks\imcoh\csd\filters\1_4\conditions\6\results\6
Empty or non-existent folder: